In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline


In [2]:
# Load trained model and encoders from Part 2
with open('../models/best_model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('../models/label_encoders.pkl', 'rb') as f:
    le_dict = pickle.load(f)

with open('../models/feature_cols.pkl', 'rb') as f:
    feature_cols = pickle.load(f)

# Load cleaned data
df = pd.read_csv('../data/raw/StudentsPerformance.csv')
df.columns = df.columns.str.replace(' ', '_').str.replace('/', '_')
df['average_score'] = df[['math_score', 'reading_score', 'writing_score']].mean(axis=1)
df['at_risk'] = (df['average_score'] < 60).astype(int)

print(f"Dataset loaded: {df.shape}")
print(f"At-risk students: {df['at_risk'].sum()} ({df['at_risk'].mean()*100:.1f}%)")

Dataset loaded: (1000, 10)
At-risk students: 285 (28.5%)


In [3]:
from langchain_core.documents import Document

# Convert dataset insights into text documents for RAG
documents = []

# Document 1 — overall stats
overall_stats = f"""
EduPulse Student Performance Overview:
- Total students: {len(df)}
- At-risk students (avg score below 60): {df['at_risk'].sum()} ({df['at_risk'].mean()*100:.1f}%)
- Average math score: {df['math_score'].mean():.1f}
- Average reading score: {df['reading_score'].mean():.1f}
- Average writing score: {df['writing_score'].mean():.1f}
"""
documents.append(Document(page_content=overall_stats, metadata={"source": "overall_stats"}))

# Document 2 — test prep impact
test_prep = df.groupby('test_preparation_course')['average_score'].mean()
test_prep_doc = f"""
Impact of Test Preparation Course on Student Performance:
- Students who completed test prep: avg score {test_prep.get('completed', 0):.1f}
- Students with no test prep: avg score {test_prep.get('none', 0):.1f}
- Difference: {abs(test_prep.get('completed', 0) - test_prep.get('none', 0)):.1f} points
- Test preparation course is the strongest predictor of at-risk status according to SHAP analysis.
- Students without test prep are significantly more likely to be flagged as at-risk.
"""
documents.append(Document(page_content=test_prep_doc, metadata={"source": "test_prep_analysis"}))

# Document 3 — lunch/socioeconomic impact
lunch = df.groupby('lunch')['average_score'].mean()
lunch_doc = f"""
Impact of Lunch Type (Socioeconomic Indicator) on Student Performance:
- Students with standard lunch: avg score {lunch.get('standard', 0):.1f}
- Students with free/reduced lunch: avg score {lunch.get('free/reduced', 0):.1f}
- Lunch type is the second strongest predictor of at-risk status.
- Students on free/reduced lunch (lower socioeconomic status) face higher academic risk.
- Recommended action: flag for financial support review and additional academic resources.
"""
documents.append(Document(page_content=lunch_doc, metadata={"source": "lunch_analysis"}))

# Document 4 — parental education impact
par_edu = df.groupby('parental_level_of_education')['average_score'].mean().sort_values()
par_edu_text = "\n".join([f"  - {k}: avg score {v:.1f}" for k, v in par_edu.items()])
par_edu_doc = f"""
Impact of Parental Education Level on Student Performance:
{par_edu_text}
- Higher parental education correlates with better student outcomes.
- Students whose parents have lower education levels may need additional mentoring support.
"""
documents.append(Document(page_content=par_edu_doc, metadata={"source": "parental_education_analysis"}))

# Document 5 — SHAP findings summary
shap_doc = """
SHAP Explainability Findings — EduPulse ML Model:
- Most important feature: test_preparation_course
    Completing test prep strongly reduces at-risk probability.
    Not completing test prep is the single biggest driver of at-risk classification.
- Second most important: lunch type (socioeconomic proxy)
    Standard lunch (higher SES) significantly reduces risk.
    Free/reduced lunch increases risk probability.
- Third: parental_level_of_education — moderate influence
- Weak predictors: gender, race/ethnicity — minimal impact on risk classification.
Key insight: Intervention should prioritise test prep enrollment and 
socioeconomic support before considering demographic factors.
"""
documents.append(Document(page_content=shap_doc, metadata={"source": "shap_findings"}))

# Document 6 — intervention recommendations
intervention_doc = """
EduPulse Intervention Recommendations by Risk Factor:

If test_preparation_course = none:
- Priority action: Enroll student in test preparation course immediately
- Expected impact: avg score improvement of 5-10 points based on dataset analysis
- Timeline: before next assessment cycle

If lunch = free/reduced (low socioeconomic status):
- Flag for financial support reviewZ
- Connect with student welfare services
- Assign peer mentor or additional tutoring sessions
- Increase check-in frequency to weekly

If parental_level_of_education = some high school or high school:
- Increase parental engagement — send regular progress reports
- Connect student with college readiness programs
- Assign academic counselor for guidance

Combined high-risk profile (no test prep + free/reduced lunch):
- Treat as priority case — immediate multi-pronged intervention
- Involve both academic and welfare support teams
"""
documents.append(Document(page_content=intervention_doc, metadata={"source": "intervention_guide"}))

print(f"Knowledge base built: {len(documents)} documents")

# Convert dataset insights into text documents for RAG
documents = []

# Document 1 — overall stats
overall_stats = f"""
EduPulse Student Performance Overview:
- Total students: {len(df)}
- At-risk students (avg score below 60): {df['at_risk'].sum()} ({df['at_risk'].mean()*100:.1f}%)
- Average math score: {df['math_score'].mean():.1f}
- Average reading score: {df['reading_score'].mean():.1f}
- Average writing score: {df['writing_score'].mean():.1f}
"""
# Convert dataset insights into text documents for RAG
documents = []

# Document 1 — overall stats
overall_stats = f"""
EduPulse Student Performance Overview:
- Total students: {len(df)}
- At-risk students (avg score below 60): {df['at_risk'].sum()} ({df['at_risk'].mean()*100:.1f}%)
- Average math score: {df['math_score'].mean():.1f}
- Average reading score: {df['reading_score'].mean():.1f}
- Average writing score: {df['writing_score'].mean():.1f}
"""
documents.append(Document(page_content=overall_stats, metadata={"source": "overall_stats"}))

# Document 2 — test prep impact
test_prep = df.groupby('test_preparation_course')['average_score'].mean()
test_prep_doc = f"""
Impact of Test Preparation Course on Student Performance:
- Students who completed test prep: avg score {test_prep.get('completed', 0):.1f}
- Students with no test prep: avg score {test_prep.get('none', 0):.1f}
- Difference: {abs(test_prep.get('completed', 0) - test_prep.get('none', 0)):.1f} points
- Test preparation course is the strongest predictor of at-risk status according to SHAP analysis.
- Students without test prep are significantly more likely to be flagged as at-risk.
"""
documents.append(Document(page_content=test_prep_doc, metadata={"source": "test_prep_analysis"}))

# Document 3 — lunch/socioeconomic impact
lunch = df.groupby('lunch')['average_score'].mean()
lunch_doc = f"""
Impact of Lunch Type (Socioeconomic Indicator) on Student Performance:
- Students with standard lunch: avg score {lunch.get('standard', 0):.1f}
- Students with free/reduced lunch: avg score {lunch.get('free/reduced', 0):.1f}
- Lunch type is the second strongest predictor of at-risk status.
- Students on free/reduced lunch (lower socioeconomic status) face higher academic risk.
- Recommended action: flag for financial support review and additional academic resources.
"""
documents.append(Document(page_content=lunch_doc, metadata={"source": "lunch_analysis"}))

# Document 4 — parental education impact
par_edu = df.groupby('parental_level_of_education')['average_score'].mean().sort_values()
par_edu_text = "\n".join([f"  - {k}: avg score {v:.1f}" for k, v in par_edu.items()])
par_edu_doc = f"""
Impact of Parental Education Level on Student Performance:
{par_edu_text}
- Higher parental education correlates with better student outcomes.
- Students whose parents have lower education levels may need additional mentoring support.
"""
documents.append(Document(page_content=par_edu_doc, metadata={"source": "parental_education_analysis"}))

# Document 5 — SHAP findings summary
shap_doc = """
SHAP Explainability Findings — EduPulse ML Model:
- Most important feature: test_preparation_course
    Completing test prep strongly reduces at-risk probability.
    Not completing test prep is the single biggest driver of at-risk classification.
- Second most important: lunch type (socioeconomic proxy)
    Standard lunch (higher SES) significantly reduces risk.
    Free/reduced lunch increases risk probability.
- Third: parental_level_of_education — moderate influence
- Weak predictors: gender, race/ethnicity — minimal impact on risk classification.
Key insight: Intervention should prioritise test prep enrollment and 
socioeconomic support before considering demographic factors.
"""
documents.append(Document(page_content=shap_doc, metadata={"source": "shap_findings"}))

# Document 6 — intervention recommendations
intervention_doc = """
EduPulse Intervention Recommendations by Risk Factor:

If test_preparation_course = none:
- Priority action: Enroll student in test preparation course immediately
- Expected impact: avg score improvement of 5-10 points based on dataset analysis
- Timeline: before next assessment cycle

If lunch = free/reduced (low socioeconomic status):
- Flag for financial support review
- Connect with student welfare services
- Assign peer mentor or additional tutoring sessions
- Increase check-in frequency to weekly

If parental_level_of_education = some high school or high school:
- Increase parental engagement — send regular progress reports
- Connect student with college readiness programs
- Assign academic counselor for guidance

Combined high-risk profile (no test prep + free/reduced lunch):
- Treat as priority case — immediate multi-pronged intervention
- Involve both academic and welfare support teams
"""
documents.append(Document(page_content=intervention_doc, metadata={"source": "intervention_guide"}))

print(f"Knowledge base built: {len(documents)} documents")

# Document 2 — test prep impact
test_prep = df.groupby('test_preparation_course')['average_score'].mean()
test_prep_doc = f"""
Impact of Test Preparation Course on Student Performance:
- Students who completed test prep: avg score {test_prep.get('completed', 0):.1f}
- Students with no test prep: avg score {test_prep.get('none', 0):.1f}
- Difference: {abs(test_prep.get('completed', 0) - test_prep.get('none', 0)):.1f} points
- Test preparation course is the strongest predictor of at-risk status according to SHAP analysis.
- Students without test prep are significantly more likely to be flagged as at-risk.
"""
documents.append(Document(page_content=test_prep_doc, metadata={"source": "test_prep_analysis"}))

# Document 3 — lunch/socioeconomic impact
lunch = df.groupby('lunch')['average_score'].mean()
lunch_doc = f"""
Impact of Lunch Type (Socioeconomic Indicator) on Student Performance:
- Students with standard lunch: avg score {lunch.get('standard', 0):.1f}
- Students with free/reduced lunch: avg score {lunch.get('free/reduced', 0):.1f}
- Lunch type is the second strongest predictor of at-risk status.
- Students on free/reduced lunch (lower socioeconomic status) face higher academic risk.
- Recommended action: flag for financial support review and additional academic resources.
"""
documents.append(Document(page_content=lunch_doc, metadata={"source": "lunch_analysis"}))

# Document 4 — parental education impact
par_edu = df.groupby('parental_level_of_education')['average_score'].mean().sort_values()
par_edu_text = "\n".join([f"  - {k}: avg score {v:.1f}" for k, v in par_edu.items()])
par_edu_doc = f"""
Impact of Parental Education Level on Student Performance:
{par_edu_text}
- Higher parental education correlates with better student outcomes.
- Students whose parents have lower education levels may need additional mentoring support.
"""
documents.append(Document(page_content=par_edu_doc, metadata={"source": "parental_education_analysis"}))

# Document 5 — SHAP findings summary
shap_doc = """
SHAP Explainability Findings — EduPulse ML Model:
- Most important feature: test_preparation_course
  Completing test prep strongly reduces at-risk probability.
  Not completing test prep is the single biggest driver of at-risk classification.
- Second most important: lunch type (socioeconomic proxy)
  Standard lunch (higher SES) significantly reduces risk.
  Free/reduced lunch increases risk probability.
- Third: parental_level_of_education — moderate influence
- Weak predictors: gender, race/ethnicity — minimal impact on risk classification.
Key insight: Intervention should prioritise test prep enrollment and 
socioeconomic support before considering demographic factors.
"""
documents.append(Document(page_content=shap_doc, metadata={"source": "shap_findings"}))

# Document 6 — intervention recommendations
intervention_doc = """
EduPulse Intervention Recommendations by Risk Factor:

If test_preparation_course = none:
- Priority action: Enroll student in test preparation course immediately
- Expected impact: avg score improvement of 5-10 points based on dataset analysis
- Timeline: before next assessment cycle

If lunch = free/reduced (low socioeconomic status):
- Flag for financial support review
- Connect with student welfare services
- Assign peer mentor or additional tutoring sessions
- Increase check-in frequency to weekly

If parental_level_of_education = some high school or high school:
- Increase parental engagement — send regular progress reports
- Connect student with college readiness programs
- Assign academic counselor for guidance

Combined high-risk profile (no test prep + free/reduced lunch):
- Treat as priority case — immediate multi-pronged intervention
- Involve both academic and welfare support teams
"""
documents.append(Document(page_content=intervention_doc, metadata={"source": "intervention_guide"}))

print(f"Knowledge base built: {len(documents)} documents")

Knowledge base built: 6 documents
Knowledge base built: 6 documents
Knowledge base built: 11 documents


In [4]:
# Use sentence-transformers for embeddings (free, runs locally)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build FAISS vector store from documents
vectorstore = FAISS.from_documents(documents, embeddings)

# Save it so you don't rebuild every time
vectorstore.save_local("../models/faiss_index")

print("Vector store created and saved.")
print(f"Indexed {vectorstore.index.ntotal} document chunks")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store created and saved.
Indexed 11 document chunks


In [ ]:
# Retriever — fetches top 3 relevant documents for any query
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def analyst_agent(question):
    """
    Answers questions about the student dataset
    by retrieving relevant knowledge base documents.
    """
    # Retrieve relevant docs
    relevant_docs = retriever.get_relevant_documents(question)
    
    # Build context from retrieved docs
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Generate answer grounded in retrieved context
    answer = f"""
ANALYST AGENT — EduPulse
========================
Question: {question}

Retrieved Context:
{context}

Answer based on EduPulse data:
"""
    # Add interpretation layer
    question_lower = question.lower()
    
    if "risk" in question_lower and "factor" in question_lower:
        answer += "The top risk factors identified by the ML model (confirmed by SHAP analysis) are:\n1. Not completing test preparation course\n2. Free/reduced lunch (socioeconomic indicator)\n3. Lower parental education level"
    
    elif "how many" in question_lower or "number" in question_lower:
        at_risk_count = df['at_risk'].sum()
        total = len(df)
        answer += f"There are {at_risk_count} at-risk students out of {total} total ({at_risk_count/total*100:.1f}%)."
    
    elif "test prep" in question_lower or "preparation" in question_lower:
        completed = df[df['test_preparation_course']=='completed']['average_score'].mean()
        none = df[df['test_preparation_course']=='none']['average_score'].mean()
        answer += f"Students who completed test prep scored {completed:.1f} on average vs {none:.1f} for those who didn't — a {abs(completed-none):.1f} point difference."
    
    elif "lunch" in question_lower or "socioeconomic" in question_lower:
        standard = df[df['lunch']=='standard']['average_score'].mean()
        reduced = df[df['lunch']=='free/reduced']['average_score'].mean()
        answer += f"Students with standard lunch scored {standard:.1f} on average vs {reduced:.1f} for free/reduced lunch — a {abs(standard-reduced):.1f} point gap."
    
    else:
        answer += context[:500] + "..."
    
    return answer

# Test the analyst agent
question = "What are the main risk factors for student performance?"
# Retriever — fetches top 3 relevant documents for any query
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def analyst_agent(question):
    """
    Answers questions about the student dataset
    by retrieving relevant knowledge base documents.
    """
    # Retrieve relevant docs
    if hasattr(retriever, "get_relevant_documents"):
        relevant_docs = retriever.get_relevant_documents(question)
    else:
        relevant_docs = vectorstore.similarity_search(question, k=3)
    
    # Build context from retrieved docs
    context = "\n\n".join([doc.page_content for doc in relevant_docs])
    
    # Generate answer grounded in retrieved context
    answer = f"""
ANALYST AGENT — EduPulse
========================
Question: {question}

Retrieved Context:
{context}

Answer based on EduPulse data:
"""
    # Add interpretation layer
    question_lower = question.lower()
    
    if "risk" in question_lower and "factor" in question_lower:
        answer += "The top risk factors identified by the ML model (confirmed by SHAP analysis) are:\n1. Not completing test preparation course\n2. Free/reduced lunch (socioeconomic indicator)\n3. Lower parental education level"
    
    elif "how many" in question_lower or "number" in question_lower:
        at_risk_count = df['at_risk'].sum()
        total = len(df)
        answer += f"There are {at_risk_count} at-risk students out of {total} total ({at_risk_count/total*100:.1f}%)."
    
    elif "test prep" in question_lower or "preparation" in question_lower:
        completed = df[df['test_preparation_course']=='completed']['average_score'].mean()
        none = df[df['test_preparation_course']=='none']['average_score'].mean()
        answer += f"Students who completed test prep scored {completed:.1f} on average vs {none:.1f} for those who didn't — a {abs(completed-none):.1f} point difference."
    
    elif "lunch" in question_lower or "socioeconomic" in question_lower:
        standard = df[df['lunch']=='standard']['average_score'].mean()
        reduced = df[df['lunch']=='free/reduced']['average_score'].mean()
        answer += f"Students with standard lunch scored {standard:.1f} on average vs {reduced:.1f} for free/reduced lunch — a {abs(standard-reduced):.1f} point gap."
    
    else:
        answer += context[:500] + "..."
    
    return answer

# Test the analyst agent
question = "What are the main risk factors for student performance?"
print(analyst_agent(question))

AttributeError: 'VectorStoreRetriever' object has no attribute 'get_relevant_documents'